# PanDerm LoRA Fine-tuning — Oral_Diseases (근거리 도메인 · 7-class · few-shot)

> 9차 미팅 `🧠 [학습] PEFT + Fine-tuning` · LoRA 파트

`Oral_Diseases` (구강, 7-class, train 407 / val 55 / test 54, 클래스당 42~74장 few-shot). 근거리 도메인 — 전략 문서가 PEFT 를 1순위로 권장하는 조건. frozen 백본에 **LoRA(순수)** 와 **LoRA+융합[11,15,19,23]** 두 변형을 학습해 Linear Evaluation baseline(BACC 0.811 / Macro AUPR 0.798) 대비 성능을 본다.

공통 로직은 [`peft_ft_utils.py`](peft_ft_utils.py) 에 있다(LoRA 구현·데이터·지표·학습 루프). 이 노트북은 얇은 래퍼다.

**실행 환경**: 집의 RTX 3070(8GB) 기준. `BATCH_SIZE=8, ACCUM=4`(유효 32), AMP, grad-checkpoint. OOM 시 `BATCH_SIZE=4` 로 낮춘다.

## 0. 설정

In [1]:
import importlib
import pandas as pd
import peft_ft_utils as U
importlib.reload(U)

DATASET = "oral_diseases"
cfg = U.DATASETS[DATASET]
print("device:", U.get_device())
print("checkpoint exists:", U.CHECKPOINT.exists())
print("classes:", cfg["class_names"])
# 소수 클래스: OLP(5) — baseline recall 0.30 개선이 핵심 목표

device: cuda
checkpoint exists: True
classes: ['CaS', 'CoS', 'Gum', 'MC', 'OC', 'OLP', 'OT']


## 1. Baseline 정합성 확인 (frozen 마지막 CLS + linear probe)

신규 방법 수치를 신뢰하기 전에, **동일 백본·transform 으로 Linear Eval baseline 을 먼저 재현**해 문서 앵커와 ±0.02 이내인지 확인한다(기존 노트북의 alignment 관례).

In [2]:
ok, got, anchor = U.reproduce_baseline(DATASET, verbose=True)
print("baseline 재현:", "OK" if ok else "MISMATCH")
assert ok, "baseline 재현 실패 — 체크포인트/transform/split 확인 필요"

/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


loading model checkpoint


/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      

100%|██████████| 1/1 [00:01<00:00,  1.11s/it]


NUM_C, Cost: 7 71.68
0
Classification report:
              precision    recall  f1-score   support

           0       1.00      0.88      0.93         8
           1       1.00      1.00      1.00         8
           2       0.70      1.00      0.82         7
           3       0.75      0.67      0.71         9
           4       0.67      1.00      0.80         6
           5       0.75      0.30      0.43        10
           6       0.62      0.83      0.71         6

    accuracy                           0.78        54
   macro avg       0.78      0.81      0.77        54
weighted avg       0.79      0.78      0.76        54

Debug: targets_all shape: (54,)
Debug: preds_all shape: (54,)
Debug: probs_all shape: (54, 7)
Debug: test_filenames length: 54
Model predicted results saved to /home/junkim/paper_ajou_dev/PanDerm/output_dir/_baseline_reproduce_tmp/oral_diseases/oral_diseases.csv
[oral_diseases] reproduce BACC=0.8107 (anchor 0.8107) AUPR=0.7981 (anchor 0.7981)
baseline 재현:

<Figure size 1000x800 with 0 Axes>

## 2. 변형 A — 순수 LoRA + linear head (마지막 CLS)

`FUSION_LAYERS=[23]` + 전 24블록 qkv/proj LoRA + `Linear(1024→K)`. Linear Eval·Full FT 와 **깨끗한 비교축**. 학습 파라미터 <1%.

In [3]:
res_pure = U.run_lora_experiment(
    DATASET, "pure",
    lora_r=8, lora_alpha=16, batch_size=8, accum_steps=4,
    epochs=40, patience=10, use_grad_checkpoint=True, seed=0,
)
print("\n[pure] test metrics:", {k: round(v,4) for k,v in res_pure["metrics"].items() if k!="per_class_recall"})
print("[pure] bootstrap CI:", res_pure["ci"])
print("[pure] 소수 클래스 recall:", res_pure["metrics"]["per_class_recall"])

loading model checkpoint
VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elemen

/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one

저장: /home/junkim/paper_ajou_dev/PanDerm/output_dir/oral_diseases_lora_lp

[pure] test metrics: {'bacc': 0.8107, 'acc': 0.7778, 'weighted_f1': 0.7653, 'auroc': np.float64(0.9655), 'aupr': 0.8609}
[pure] bootstrap CI: {'bacc_ci': (0.7213690476190476, 0.8928996598639456), 'aupr_ci': (0.7696279316785691, 0.9518364399631907)}
[pure] 소수 클래스 recall: {'0': 0.875, '1': 1.0, '2': 1.0, '3': 0.6666666666666666, '4': 1.0, '5': 0.3, '6': 0.8333333333333334}


## 3. 변형 B — LoRA + 멀티레이어 융합 (ablation)

`FUSION_LAYERS=[11,15,19,23]` (중간층 결합). 융합이 성능에 기여하는지 분리 확인.

In [4]:
res_fusion = U.run_lora_experiment(
    DATASET, "fusion",
    lora_r=8, lora_alpha=16, batch_size=8, accum_steps=4,
    epochs=40, patience=10, use_grad_checkpoint=True, seed=0,
)
print("\n[fusion] test metrics:", {k: round(v,4) for k,v in res_fusion["metrics"].items() if k!="per_class_recall"})
print("[fusion] bootstrap CI:", res_fusion["ci"])
print("[fusion] 소수 클래스 recall:", res_fusion["metrics"]["per_class_recall"])

loading model checkpoint
VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elemen

/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one

저장: /home/junkim/paper_ajou_dev/PanDerm/output_dir/oral_diseases_lora_multilayer_lp

[fusion] test metrics: {'bacc': 0.7948, 'acc': 0.7778, 'weighted_f1': 0.7777, 'auroc': np.float64(0.9492), 'aupr': 0.8163}
[fusion] bootstrap CI: {'bacc_ci': (0.695912480376766, 0.8923044217687073), 'aupr_ci': (0.7332289569053427, 0.9182207486006126)}
[fusion] 소수 클래스 recall: {'0': 0.875, '1': 1.0, '2': 1.0, '3': 0.8888888888888888, '4': 0.6666666666666666, '5': 0.3, '6': 0.8333333333333334}


## 4. 비교표 (baseline vs LoRA-순수 vs LoRA-융합)

지표: BACC / Macro AUPR / AUROC(ovo) / W-F1 / Acc. 모두 동일 test split·동일 지표 정의.

In [5]:
base = res_pure["baseline"]
def row(name, m):
    return {"method": name, "BACC": m["bacc"], "Macro_AUPR": m["aupr"],
            "AUROC": m["auroc"], "W_F1": m["weighted_f1"], "ACC": m["acc"]}
table = pd.DataFrame([
    {"method": "Linear Eval (baseline)", "BACC": base["bacc"], "Macro_AUPR": base["aupr"],
     "AUROC": base["auroc"], "W_F1": base["weighted_f1"], "ACC": base["acc"]},
    row("LoRA (pure, last CLS)", res_pure["metrics"]),
    row("LoRA (fusion [11,15,19,23])", res_fusion["metrics"]),
]).set_index("method").round(4)
print(table)
table.to_csv(U.OUTPUT_ROOT / f"{DATASET}_lora_summary.csv")
table

                               BACC  Macro_AUPR   AUROC    W_F1     ACC
method                                                                 
Linear Eval (baseline)       0.8107      0.7981  0.9463  0.7584  0.7778
LoRA (pure, last CLS)        0.8107      0.8609  0.9655  0.7653  0.7778
LoRA (fusion [11,15,19,23])  0.7948      0.8163  0.9492  0.7777  0.7778


,BACC,Macro_AUPR,AUROC,W_F1,ACC
method,,,,,
Linear Eval (baseline),0.8107,0.7981,0.9463,0.7584,0.7778
"LoRA (pure, last CLS)",0.8107,0.8609,0.9655,0.7653,0.7778
"LoRA (fusion [11,15,19,23])",0.7948,0.8163,0.9492,0.7777,0.7778


---
**저장 위치**
- 순수: `PanDerm/output_dir/oral_diseases_lora_lp/`
- 융합: `PanDerm/output_dir/oral_diseases_lora_multilayer_lp/`
- 요약: `PanDerm/output_dir/oral_diseases_lora_summary.csv`

각 폴더에 `training_history.csv`, `best_trainable_state_dict.pt`, `comparison_vs_baseline.csv`, `*_test_predictions.csv`.